In [1]:
import torch

d:\Kế hoạch du học\Deap_learning_course\Residuals_Normalizations\venv\Lib\site-packages\torch\_subclasses\functional_tensor.py:377: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:82.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


In [2]:
class MyModelLN(torch.nn.Module):
    class Block(torch.nn.Module):
        def __init__(self, in_channels, out_channels):
            super().__init__()
            self.model = torch.nn.Sequential(
                torch.nn.Linear(in_channels, out_channels),
                torch.nn.LayerNorm(out_channels),
                torch.nn.ReLU(),
                torch.nn.Linear(out_channels, out_channels),
                torch.nn.LayerNorm(out_channels),
                torch.nn.ReLU(),
            )
            if in_channels != out_channels:
                self.skip = torch.nn.Linear(in_channels, out_channels)
            else:
                self.skip = torch.nn.Identity()

        def forward(self, x):
            return self.skip(x) + self.model(x)

    def __init__(self, layer_size=[512, 512, 512]):
        super().__init__()
        layers = []
        layers.append(torch.nn.Flatten())
        c = 128 * 128 * 3
        layers.append(torch.nn.Linear(c, 512, bias=False))
        c = 512
        for s in layer_size:
            layers.append(self.Block(c, s))
            c = s
        layers.append(torch.nn.Linear(c, 102, bias=False))
        self.model = torch.nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)


x = torch.randn(10, 3, 128, 128)
net = MyModelLN([512] * 4)
print(net(x))

tensor([[-0.8020,  0.2901, -0.8813,  ..., -0.5477, -2.4004,  0.7495],
        [ 0.0208, -1.5758, -0.7983,  ..., -0.8541, -1.4180, -0.3411],
        [-1.4989, -0.1349, -0.0408,  ..., -0.5397, -2.9015, -0.5982],
        ...,
        [-1.6523,  0.9501, -0.8562,  ..., -0.8751, -1.2766, -0.5105],
        [-2.0830, -0.6973, -0.8684,  ...,  0.0487, -1.1222, -0.4723],
        [-1.5336,  0.6368,  0.2479,  ...,  0.6437, -1.9652, -0.3007]],
       grad_fn=<MmBackward0>)


In [3]:
128 * 128 * 3

49152